In [82]:
# importación de todas las librerías necesarias
import pandas as pd
import ast
import numpy as np

En este archivo se prepara definitivamente el dataset final de la segunda entrega, juntando todos los indicadores y procesos realizados para convertir el conjunto de datos original a un dataset con todo datos numéricos. 

---

1. **Proceso del resultado del modelo zeroshot.** Convertimos el diccionario alojado en la columna de resultados *"modelo_zeroshot"* en varias columnas, una por cada categoría con su valor asociado a dicha categoría para cada titular. 

In [83]:
# Cargamos todos los tres datasets y combinamos todos los datos (reconstruimos el conjunto entero)
df1 = pd.read_csv("data_primer_tercio_zeroshot.csv", index_col="Unnamed: 0")  # Dado que se han guardado los conjuntos con IDs como columna aparte la convertimos en el ID al cargar
df2 = pd.read_csv("data_segundo_tercio_zeroshot.csv", index_col="Unnamed: 0")  
df3 = pd.read_csv("data_tercer_tercio_zeroshot.csv", index_col="Unnamed: 0")

df = pd.concat([df1, df2, df3], axis=0)
print(f"Dimensiones del conjunto: {df.shape[0]} filas x {df.shape[1]} columnas.")

Dimensiones del conjunto: 162574 filas x 19 columnas.


In [ ]:
# Función de expansión de la columna diccionario

def expandir_diccionarios(df, columna_diccionario):
    """
    Alternativa usando json_normalize con protección contra duplicados
    """
    df_resultado = df.copy()
    
    try:
        # Convertir a diccionarios
        dicts = df_resultado[columna_diccionario].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) and x not in ["", "{}"] else {}
        )
        
        # Usar json_normalize pero asegurar índices únicos
        df_expandido = pd.json_normalize(dicts)
        
        # VERIFICACIÓN CRÍTICA: eliminar columnas duplicadas
        df_expandido = df_expandido.loc[:, ~df_expandido.columns.duplicated()]
        
        # Resetear índices para asegurar compatibilidad
        df_resultado = df_resultado.reset_index(drop=True)
        df_expandido = df_expandido.reset_index(drop=True)
        
        # Combinar
        df_resultado = pd.concat([df_resultado, df_expandido], axis=1)
        
        print(f"✓ Expansión exitosa. Columnas añadidas: {list(df_expandido.columns)}")
        
    except Exception as e:
        print(f"Error en json_normalize: {e}")
        
        # Fallback a método manual
        print("Intentando con método manual...")
        df_resultado = expandir_diccionarios_simple(df, columna_diccionario)
    
    return df_resultado

df_expand = expandir_diccionarios(df, "modelo_zeroshot")  # reasociar variable??
print(f"Dimensiones del conjunto: {df_expand.shape[0]} filas x {df_expand.shape[1]} columnas.")

✓ Expansión exitosa. Columnas añadidas: ['empleo', 'economía', 'tecnología', 'salud']
Dimensiones del conjunto: 162574 filas x 23 columnas.


In [85]:
df_expand.head()

,ID_Disposicion,Fecha_Publicacion,Titulo_Semantico,Año_Semana,Año,Bloque_3dias,texto_limpio,texto_lower,Mes,Trimestre,...,longitud_titulo,processed_tokens,processed_text,quinquenio,text_str,modelo_zeroshot,empleo,economía,tecnología,salud
0,BOE-A-1995-26,1995-01-02,"Resolución de 30 de noviembre de 1994, de la U...",1995-01,1995,0,Resolucion de de noviembre de de la Universida...,resolucion de de noviembre de de la universida...,1,1995Q1,...,30,"['resolucion', 'politecnico', 'madrid', 'convo...",resolucion politecnico madrid convocar libre d...,1995,resolucion politecnico madrid convocar libre d...,"{'empleo': 0.9980953335762024, 'economía': 3.0...",0.998095,0.000030,0.000021,0.000017
1,BOE-A-1995-27,1995-01-02,"Resolución de 12 de diciembre de 1994, de la U...",1995-01,1995,0,Resolucion de de diciembre de de la Universida...,resolucion de de diciembre de de la universida...,1,1995Q1,...,42,"['resolucion', 'politecnico', 'madrid', 'convo...",resolucion politecnico madrid convocar libre d...,1995,resolucion politecnico madrid convocar libre d...,"{'empleo': 0.9998026490211487, 'tecnología': 0...",0.999803,0.000024,0.056125,0.000014
2,BOE-A-1995-77,1995-01-03,"Resolución de 28 de noviembre de 1994, de la U...",1995-01,1995,0,Resolucion de de noviembre de de la Universida...,resolucion de de noviembre de de la universida...,1,1995Q1,...,48,"['resolucion', 'complutense', 'madrid', 'miemb...",resolucion complutense madrid miembro comisión...,1995,resolucion complutense madrid miembro comisión...,"{'empleo': 0.9715206623077393, 'salud': 0.0080...",0.971521,0.000025,0.000023,0.008056
3,BOE-A-1995-78,1995-01-03,"Resolución de 13 de diciembre de 1994, de la U...",1995-01,1995,0,Resolucion de de diciembre de de la Universida...,resolucion de de diciembre de de la universida...,1,1995Q1,...,40,"['resolucion', 'jaen', 'composicion', 'comisió...",resolucion jaen composicion comisión resolver ...,1995,resolucion jaen composicion comisión resolver ...,"{'empleo': 0.0017847735434770584, 'tecnología'...",0.001785,0.000021,0.000024,0.000014
4,BOE-A-1995-205,1995-01-03,"Resolución de 12 de diciembre de 1994, de la U...",1995-01,1995,0,Resolucion de de diciembre de de la Universida...,resolucion de de diciembre de de la universida...,1,1995Q1,...,45,"['resolucion', 'santiago', 'compostela', 'corr...",resolucion santiago compostela corregir error ...,1995,resolucion santiago compostela corregir error ...,"{'economía': 1.5204033843474463e-05, 'tecnolog...",0.000014,0.000015,0.000015,0.000013


---

2. Unión con dataset derivado del indicador sobre si la universidad mencionada es pública o privada. 

In [86]:
# Cargamos datasets tipo de entidad 
df_publi_priv = pd.read_csv("boe_universidades_publica_priv.csv")
print(f"Dimensiones del dataset derivado del indicador de tipo de entidad: {df_publi_priv.shape[0]} filas x {df_publi_priv.shape[1]} columnas.")

Dimensiones del dataset derivado del indicador de tipo de entidad: 162574 filas x 17 columnas.


In [87]:
# Unimos el dataset derivado del zeroshot con el de tipo de entidad, añadiendo solo columnas que no existan
df_publi_priv_zeroshot = df_expand.copy()
nuevas_columnas = [col for col in df_publi_priv.columns if col not in df_expand.columns]

if nuevas_columnas:
    df_publi_priv_zeroshot = pd.merge(df_expand, df_publi_priv[['ID_Disposicion'] + nuevas_columnas], on='ID_Disposicion', how='left')
    
print(f"Dimensiones del dataset derivado del indicador de tipo de entidad + zeroshot: {df_publi_priv_zeroshot.shape[0]} filas x {df_publi_priv_zeroshot.shape[1]} columnas.")

Dimensiones del dataset derivado del indicador de tipo de entidad + zeroshot: 162574 filas x 26 columnas.


In [88]:
df_publi_priv_zeroshot.columns

Index(['ID_Disposicion', 'Fecha_Publicacion', 'Titulo_Semantico', 'Año_Semana',
       'Año', 'Bloque_3dias', 'texto_limpio', 'texto_lower', 'Mes',
       'Trimestre', 'Bloques_dias_semana', 'Dia_semana_num', 'Dia_semana',
       'longitud_titulo', 'processed_tokens', 'processed_text', 'quinquenio',
       'text_str', 'modelo_zeroshot', 'empleo', 'economía', 'tecnología',
       'salud', 'universidades_mencionadas', 'publica', 'privada'],
      dtype='object')

---

3. Unión con el dataset derivado del análisis de las provincias

In [89]:
# Cargamos el conjunto de datos derivado del indicador de provincia
df_provincias = pd.read_csv("dataset_universidades_provincias.csv")
print(f"Dimensiones del dataset de provincias: {df_provincias.shape[0]} filas x {df_provincias.shape[1]} columnas.")

Dimensiones del dataset de provincias: 162574 filas x 50 columnas.


In [90]:
# Unimos el dataset con el derivado del estudio de provincias, añadiendo solo columnas que no existan
df_fin = df_publi_priv_zeroshot.copy()
nuevas_columnas = [col for col in df_provincias.columns if col not in df_publi_priv_zeroshot.columns]

if nuevas_columnas:
    df_fin = pd.merge(df_publi_priv_zeroshot, df_provincias[['ID_Disposicion'] + nuevas_columnas], on='ID_Disposicion', how='left')
    
print(f"Dimensiones del dataset derivado del indicador de tipo de entidad + zeroshot: {df_fin.shape[0]} filas x {df_fin.shape[1]} columnas.")

Dimensiones del dataset derivado del indicador de tipo de entidad + zeroshot: 162574 filas x 62 columnas.


In [91]:
df_fin.columns

Index(['ID_Disposicion', 'Fecha_Publicacion', 'Titulo_Semantico', 'Año_Semana',
       'Año', 'Bloque_3dias', 'texto_limpio', 'texto_lower', 'Mes',
       'Trimestre', 'Bloques_dias_semana', 'Dia_semana_num', 'Dia_semana',
       'longitud_titulo', 'processed_tokens', 'processed_text', 'quinquenio',
       'text_str', 'modelo_zeroshot', 'empleo', 'economía', 'tecnología',
       'salud', 'universidades_mencionadas', 'publica', 'privada', 'Sevilla',
       'Granada', 'Córdoba', 'Málaga', 'Cádiz', 'Almería', 'Huelva', 'Jaén',
       'Zaragoza', 'Asturias', 'Islas Baleares', 'Santa Cruz de Tenerife',
       'Las Palmas', 'Cantabria', 'Salamanca', 'Valladolid', 'León', 'Burgos',
       'Ávila', 'Ciudad Real', 'Barcelona', 'Girona', 'Lleida', 'Tarragona',
       'Valencia', 'Alicante', 'Castellón', 'Badajoz', 'A Coruña',
       'Pontevedra', 'Madrid', 'Murcia', 'Navarra', 'Bizkaia', 'Gipuzkoa',
       'La Rioja'],
      dtype='object')

In [92]:
# Eliminamos las columnas de texto que ya no queremos para conseguir nuestro conjunto exclusivamente numérico
df_fin.drop(columns=['Titulo_Semantico', 'texto_limpio', 'texto_lower',
       'Trimestre', 'Dia_semana', 'processed_tokens', 'processed_text',
       'text_str', 'modelo_zeroshot', 'universidades_mencionadas'], inplace=True)

Y comprobamos los tipos de datos que contiene nuestro conjunto final tras la eliminación.

In [93]:
df_fin.dtypes

ID_Disposicion             object
Fecha_Publicacion          object
Año_Semana                 object
Año                         int64
Bloque_3dias                int64
Mes                         int64
Bloques_dias_semana         int64
Dia_semana_num              int64
longitud_titulo             int64
quinquenio                  int64
empleo                    float64
economía                  float64
tecnología                float64
salud                     float64
publica                     int64
privada                     int64
Sevilla                     int64
Granada                     int64
Córdoba                     int64
Málaga                      int64
Cádiz                       int64
Almería                     int64
Huelva                      int64
Jaén                        int64
Zaragoza                    int64
Asturias                    int64
Islas Baleares              int64
Santa Cruz de Tenerife      int64
Las Palmas                  int64
Cantabria     

In [94]:
df_fin.head()

,ID_Disposicion,Fecha_Publicacion,Año_Semana,Año,Bloque_3dias,Mes,Bloques_dias_semana,Dia_semana_num,longitud_titulo,quinquenio,...,Castellón,Badajoz,A Coruña,Pontevedra,Madrid,Murcia,Navarra,Bizkaia,Gipuzkoa,La Rioja
0,BOE-A-1995-26,1995-01-02,1995-01,1995,0,1,0,0,30,1995,...,0,0,0,0,1,0,0,0,0,0
1,BOE-A-1995-27,1995-01-02,1995-01,1995,0,1,0,0,42,1995,...,0,0,0,0,1,0,0,0,0,0
2,BOE-A-1995-77,1995-01-03,1995-01,1995,0,1,0,1,48,1995,...,0,0,0,0,1,0,0,0,0,0
3,BOE-A-1995-78,1995-01-03,1995-01,1995,0,1,0,1,40,1995,...,0,0,0,0,0,0,0,0,0,0
4,BOE-A-1995-205,1995-01-03,1995-01,1995,0,1,0,1,45,1995,...,0,0,1,0,0,0,0,0,0,0


In [95]:
# Guardamos el dataset final 
df_fin.to_csv("dataset_boe_universidades_processed.csv")

---

### **ANÁLISIS PARA LA FORMULACIÓN DE UNA HIPÓTESIS**

Una vez tenemos el conjunto de datos procesado con todos nuestros indicadores para extraer información numérica y útil de los textos, analizamos las apariciones de las diferentes combinaciones de valores por periodos de tiempo de cara a poder formular una hipótesis en base a nuestros datos. 

In [96]:
def analizar_combinaciones_provinciales(df):
    """
    Análisis de combinaciones temática-tipo de universidad-provincia por quinquenio
    """
    
    categorias = ['empleo', 'economía', 'tecnología', 'salud']
    sectores = ['publica', 'privada']
    provincias = ['Sevilla', 'Granada', 'Córdoba', 'Málaga', 'Cádiz', 'Almería', 'Huelva', 'Jaén', 
                  'Zaragoza', 'Asturias', 'Islas Baleares', 'Santa Cruz de Tenerife', 'Las Palmas', 
                  'Cantabria', 'Salamanca', 'Valladolid', 'León', 'Burgos', 'Ávila', 'Ciudad Real', 
                  'Barcelona', 'Girona', 'Lleida', 'Tarragona', 'Valencia', 'Alicante', 'Castellón', 
                  'Badajoz', 'A Coruña', 'Pontevedra', 'Madrid', 'Murcia', 'Navarra', 'Bizkaia', 
                  'Gipuzkoa', 'La Rioja']
    
    # Verificar columnas
    columnas_necesarias = categorias + sectores + provincias + ['quinquenio']
    columnas_faltantes = [col for col in columnas_necesarias if col not in df.columns]
    
    if columnas_faltantes:
        print(f"Columnas faltantes: {columnas_faltantes}")
        return None
    
    # Filtrar solo filas donde al menos un sector tenga 1 y al menos una provincia tenga 1
    mascara_sector = (df[sectores] == 1).any(axis=1)
    mascara_provincia = (df[provincias] == 1).any(axis=1)
    df_filtrado = df[mascara_sector & mascara_provincia].copy()
    
    print(f"Filas analizables con provincia: {len(df_filtrado)} de {len(df)}")
    
    def identificar_combinaciones_completas(fila):
        resultados = {}
        
        # Categoría con MAYOR valor
        valores_validos = {cat: fila[cat] for cat in categorias if not pd.isna(fila[cat])}
        if valores_validos:
            cat_mayor = max(valores_validos, key=valores_validos.get)
        else:
            cat_mayor = None
        
        # Categoría con MENOR valor (excluyendo la categoría mayor)
        categorias_restantes = [cat for cat in categorias if cat != cat_mayor and not pd.isna(fila[cat])]
        
        if categorias_restantes:
            cat_menor = min(categorias_restantes, key=lambda x: fila[x])
        else:
            cat_menor = None
        
        # Sector activo
        sector_activo = None
        for sector in sectores:
            if fila[sector] == 1:
                sector_activo = sector
                break
        
        # Provincia mencionada
        provincia_mencionada = None
        for provincia in provincias:
            if fila[provincia] == 1:
                provincia_mencionada = provincia
                break
        
        resultados['categoria_mayor'] = cat_mayor
        resultados['categoria_menor'] = cat_menor
        resultados['sector_activo'] = sector_activo
        resultados['provincia_mencionada'] = provincia_mencionada
        
        return resultados
    
    # Aplicar la función
    combinaciones = df_filtrado.apply(identificar_combinaciones_completas, axis=1, result_type='expand')
    df_combinaciones = pd.concat([df_filtrado, combinaciones], axis=1)
    
    # Filtrar filas con combinaciones válidas
    mascara_valida = (df_combinaciones['categoria_mayor'].notna()) & (df_combinaciones['sector_activo'].notna()) & (df_combinaciones['provincia_mencionada'].notna())
    df_valido = df_combinaciones[mascara_valida].copy()
    
    print(f"Filas con combinaciones completas válidas: {len(df_valido)}")
    
    # Mostrar distribución general
    print("\nDISTRIBUCIÓN GENERAL DE COMBINACIONES:")
    print("=" * 50)
    
    print("\nCategorías mayores:")
    distrib_mayor = df_valido['categoria_mayor'].value_counts()
    for cat, count in distrib_mayor.items():
        porcentaje = (count / len(df_valido)) * 100
        print(f"  {cat}: {count} ({porcentaje:.1f}%)")
    
    print("\nSectores:")
    distrib_sector = df_valido['sector_activo'].value_counts()
    for sector, count in distrib_sector.items():
        porcentaje = (count / len(df_valido)) * 100
        print(f"  {sector}: {count} ({porcentaje:.1f}%)")
    
    print("\nTop 10 provincias más mencionadas:")
    distrib_provincias = df_valido['provincia_mencionada'].value_counts().head(10)
    for prov, count in distrib_provincias.items():
        porcentaje = (count / len(df_valido)) * 100
        print(f"  {prov}: {count} ({porcentaje:.1f}%)")
    
    return df_valido

def analizar_tendencias_provinciales_por_quinquenio(df_combinaciones):
    """
    Analiza tendencias de combinaciones temática-sector-provincia por quinquenio
    """
    if df_combinaciones is None or df_combinaciones.empty:
        print("No hay datos válidos para analizar")
        return None, None
    
    resultados_mayor = []
    resultados_menor = []
    
    quinquenios = sorted(df_combinaciones['quinquenio'].unique())
    
    print(f"\nAnalizando {len(quinquenios)} quinquenios...")
    
    for quinquenio in quinquenios:
        df_q = df_combinaciones[df_combinaciones['quinquenio'] == quinquenio]
        
        print(f"\nQuinquenio {quinquenio}: {len(df_q)} combinaciones válidas")
        
        # ANÁLISIS DE COMBINACIONES CON CATEGORÍA MAYOR
        comb_mayor = df_q.groupby(['categoria_mayor', 'sector_activo', 'provincia_mencionada']).size().reset_index(name='frecuencia')
        
        if not comb_mayor.empty:
            # Top 5 combinaciones para categoría mayor
            top_mayor = comb_mayor.nlargest(5, 'frecuencia')
            
            for _, row in top_mayor.iterrows():
                resultados_mayor.append({
                    'quinquenio': quinquenio,
                    'tipo': 'mayor_valor',
                    'categoria': row['categoria_mayor'],
                    'sector': row['sector_activo'],
                    'provincia': row['provincia_mencionada'],
                    'frecuencia': row['frecuencia'],
                    'porcentaje': (row['frecuencia'] / len(df_q)) * 100
                })
        
        # ANÁLISIS DE COMBINACIONES CON CATEGORÍA MENOR
        df_q_menor = df_q[df_q['categoria_menor'].notna()]
        
        if not df_q_menor.empty:
            comb_menor = df_q_menor.groupby(['categoria_menor', 'sector_activo', 'provincia_mencionada']).size().reset_index(name='frecuencia')
            
            if not comb_menor.empty:
                # Top 5 combinaciones para categoría menor
                top_menor = comb_menor.nlargest(5, 'frecuencia')
                
                for _, row in top_menor.iterrows():
                    resultados_menor.append({
                        'quinquenio': quinquenio,
                        'tipo': 'menor_valor',
                        'categoria': row['categoria_menor'],
                        'sector': row['sector_activo'],
                        'provincia': row['provincia_mencionada'],
                        'frecuencia': row['frecuencia'],
                        'porcentaje': (row['frecuencia'] / len(df_q_menor)) * 100
                    })
    
    df_mayor = pd.DataFrame(resultados_mayor)
    df_menor = pd.DataFrame(resultados_menor)
    
    return df_mayor, df_menor

def visualizar_tendencias_provinciales_temporales(df_mayor, df_menor):
    """
    Muestra las tendencias temporales de combinaciones temática-sector-provincia
    """
    print("\n" + "=" * 100)
    print("ANÁLISIS DE TENDENCIAS TEMPORALES - COMBINACIONES TEMÁTICA-SECTOR-PROVINCIA")
    print("=" * 100)
    
    quinquenios = sorted(df_mayor['quinquenio'].unique())
    
    # TENDENCIAS PARA CATEGORÍA CON MAYOR VALOR
    print("\nCOMBINACIONES MÁS FRECUENTES - CATEGORÍA CON MAYOR VALOR:")
    print("=" * 80)
    
    for quinquenio in quinquenios:
        df_q = df_mayor[df_mayor['quinquenio'] == quinquenio]
        
        if not df_q.empty:
            print(f"\nQUINQUENIO {quinquenio}:")
            print("-" * 50)
            
            for i, (_, row) in enumerate(df_q.iterrows(), 1):
                print(f"  {i}. {row['categoria']} + {row['sector']} + {row['provincia']}")
                print(f"     Frecuencia: {row['frecuencia']} ({row['porcentaje']:.1f}%)")
    
    # TENDENCIAS PARA CATEGORÍA CON MENOR VALOR
    if not df_menor.empty:
        print("\n\nCOMBINACIONES MÁS FRECUENTES - CATEGORÍA CON MENOR VALOR:")
        print("=" * 80)
        
        for quinquenio in quinquenios:
            df_q = df_menor[df_menor['quinquenio'] == quinquenio]
            
            if not df_q.empty:
                print(f"\nQUINQUENIO {quinquenio}:")
                print("-" * 50)
                
                for i, (_, row) in enumerate(df_q.iterrows(), 1):
                    print(f"  {i}. {row['categoria']} + {row['sector']} + {row['provincia']}")
                    print(f"     Frecuencia: {row['frecuencia']} ({row['porcentaje']:.1f}%)")
    
    # ANÁLISIS DE EVOLUCIÓN TEMPORAL
    print("\n\nEVOLUCIÓN TEMPORAL DE COMBINACIONES PRINCIPALES:")
    print("=" * 60)
    
    # Para categoría mayor
    print("\nEVOLUCIÓN - CATEGORÍA MAYOR:")
    evolucion_mayor = {}
    
    for quinquenio in quinquenios:
        df_q = df_mayor[df_mayor['quinquenio'] == quinquenio]
        if not df_q.empty:
            combinacion_principal = f"{df_q.iloc[0]['categoria']} + {df_q.iloc[0]['sector']} + {df_q.iloc[0]['provincia']}"
            frecuencia_principal = df_q.iloc[0]['frecuencia']
            evolucion_mayor[quinquenio] = (combinacion_principal, frecuencia_principal)
    
    for quinquenio, (combinacion, frecuencia) in evolucion_mayor.items():
        print(f"  {quinquenio}: {combinacion} ({frecuencia} apariciones)")
    
    # Para categoría menor (si existe)
    if not df_menor.empty:
        print("\nEVOLUCIÓN - CATEGORÍA MENOR:")
        evolucion_menor = {}
        
        for quinquenio in quinquenios:
            df_q = df_menor[df_menor['quinquenio'] == quinquenio]
            if not df_q.empty:
                combinacion_principal = f"{df_q.iloc[0]['categoria']} + {df_q.iloc[0]['sector']} + {df_q.iloc[0]['provincia']}"
                frecuencia_principal = df_q.iloc[0]['frecuencia']
                evolucion_menor[quinquenio] = (combinacion_principal, frecuencia_principal)
        
        for quinquenio, (combinacion, frecuencia) in evolucion_menor.items():
            print(f"  {quinquenio}: {combinacion} ({frecuencia} apariciones)")
    
    # ANÁLISIS COMPARATIVO FINAL
    print("\n\nRESUMEN COMPARATIVO:")
    print("=" * 40)
    
    if len(quinquenios) >= 2:
        primer_q = quinquenios[0]
        ultimo_q = quinquenios[-1]
        
        if primer_q in evolucion_mayor and ultimo_q in evolucion_mayor:
            combo_inicial = evolucion_mayor[primer_q][0]
            combo_final = evolucion_mayor[ultimo_q][0]
            
            print(f"\nCATEGORÍA MAYOR:")
            print(f"  Inicio ({primer_q}): {combo_inicial}")
            print(f"  Final ({ultimo_q}): {combo_final}")
            print(f"  Cambio: {'SI' if combo_inicial != combo_final else 'NO'}")
        
        if evolucion_menor and primer_q in evolucion_menor and ultimo_q in evolucion_menor:
            combo_inicial_menor = evolucion_menor[primer_q][0]
            combo_final_menor = evolucion_menor[ultimo_q][0]
            
            print(f"\nCATEGORÍA MENOR:")
            print(f"  Inicio ({primer_q}): {combo_inicial_menor}")
            print(f"  Final ({ultimo_q}): {combo_final_menor}")
            print(f"  Cambio: {'SI' if combo_inicial_menor != combo_final_menor else 'NO'}")

def analizar_patrones_especificos_provinciales(df_combinaciones):
    """
    Analiza patrones específicos por provincia y genera hipótesis
    """
    if df_combinaciones is None or df_combinaciones.empty:
        return
    
    print("\n" + "=" * 100)
    print("ANÁLISIS DE PATRONES ESPECÍFICOS POR PROVINCIA")
    print("=" * 100)
    
    # Provincias más frecuentes
    top_provincias = df_combinaciones['provincia_mencionada'].value_counts().head(8)
    
    print("\nPATRONES POR PROVINCIA MÁS MENCIONADA:")
    print("=" * 50)
    
    for provincia in top_provincias.index:
        df_prov = df_combinaciones[df_combinaciones['provincia_mencionada'] == provincia]
        
        print(f"\n{provincia.upper()} (Total: {len(df_prov)} menciones):")
        print("-" * 40)
        
        # Distribución por categoría mayor
        print("  Temáticas principales:")
        cat_distrib = df_prov['categoria_mayor'].value_counts()
        for cat, count in cat_distrib.items():
            porcentaje = (count / len(df_prov)) * 100
            print(f"    - {cat}: {count} ({porcentaje:.1f}%)")
        
        # Distribución por sector
        print("  Distribución por sector:")
        sector_distrib = df_prov['sector_activo'].value_counts()
        for sector, count in sector_distrib.items():
            porcentaje = (count / len(df_prov)) * 100
            print(f"    - {sector}: {count} ({porcentaje:.1f}%)")
        
        # Combinación más frecuente
        combo_frecuente = df_prov.groupby(['categoria_mayor', 'sector_activo']).size().nlargest(1)
        if not combo_frecuente.empty:
            combo = combo_frecuente.index[0]
            count = combo_frecuente.iloc[0]
            print(f"  Combinación más frecuente: {combo[0]} + {combo[1]} ({count} veces)")

# EJECUCIÓN COMPLETA DEL ANÁLISIS PROVINCIAL
def ejecutar_analisis_provincial_completo(df):
    """
    Ejecuta el análisis completo de combinaciones temática-sector-provincia
    """
    print("INICIANDO ANÁLISIS DE COMBINACIONES TEMÁTICA-SECTOR-PROVINCIA...")
    print("=" * 80)
    
    # 1. Identificar combinaciones completas
    df_combinaciones = analizar_combinaciones_provinciales(df)
    
    if df_combinaciones is None or df_combinaciones.empty:
        print("No hay datos suficientes para el análisis provincial")
        return None, None, None
    
    # 2. Analizar tendencias por quinquenio
    df_mayor, df_menor = analizar_tendencias_provinciales_por_quinquenio(df_combinaciones)
    
    # 3. Visualizar tendencias
    visualizar_tendencias_provinciales_temporales(df_mayor, df_menor)
    
    # 4. Analizar patrones específicos
    analizar_patrones_especificos_provinciales(df_combinaciones)


ejecutar_analisis_provincial_completo(df_fin)

INICIANDO ANÁLISIS DE COMBINACIONES TEMÁTICA-SECTOR-PROVINCIA...
Filas analizables con provincia: 158659 de 162574
Filas con combinaciones completas válidas: 158659

DISTRIBUCIÓN GENERAL DE COMBINACIONES:

Categorías mayores:
  empleo: 107735 (67.9%)
  tecnología: 27327 (17.2%)
  salud: 12786 (8.1%)
  economía: 10811 (6.8%)

Sectores:
  publica: 151898 (95.7%)
  privada: 6761 (4.3%)

Top 10 provincias más mencionadas:
  Madrid: 38897 (24.5%)
  Valencia: 11804 (7.4%)
  Barcelona: 9518 (6.0%)
  Sevilla: 7626 (4.8%)
  Granada: 7564 (4.8%)
  A Coruña: 5245 (3.3%)
  Alicante: 4971 (3.1%)
  Bizkaia: 4940 (3.1%)
  Zaragoza: 4861 (3.1%)
  Murcia: 4410 (2.8%)

Analizando 6 quinquenios...

Quinquenio 1995: 27274 combinaciones válidas

Quinquenio 2000: 31029 combinaciones válidas

Quinquenio 2005: 22774 combinaciones válidas

Quinquenio 2010: 26788 combinaciones válidas

Quinquenio 2015: 28606 combinaciones válidas

Quinquenio 2020: 22188 combinaciones válidas

ANÁLISIS DE TENDENCIAS TEMPORALES -